# Wine Reviews — Exploratory Data Analysis

Purely exploratory notebook, not part of the final pipeline.

**Goals**
- Map missing values across columns
- Inspect distributions of `retail` (target) and `rating`
- Surface top categorical values (country, wine_type, varietal, ...)
- Compute numeric correlations
- Visualize price vs rating

Loaded from the cleaned **Silver** parquet — proper dtypes already cast,
rows with `rating < 80` dropped, zeros in `retail`/`case_production`
nulled, `country` merged, `designation` filled. See `02_cleaning.ipynb`
for the authoritative cleaning logic.

In [36]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

SILVER_PATH = r"..\..\.data\wine_reviews_silver.parquet"

df = pd.read_parquet(SILVER_PATH)
print(f"Shape: {df.shape}")
df.dtypes

Shape: (135192, 24)


wine_id              int64
name                   str
brand                  str
company                str
vintage            float64
drink_type             str
wine_type              str
varietal_label         str
alcohol            float64
bottle_size        float64
case_production    float64
country                str
state                  str
appellation            str
designation            str
retail             float64
rating               int64
reviewer               str
review                 str
date_of_review         str
date_received          str
pub_date_web           str
slug                   str
is_nv                int64
dtype: object

## Numeric columns

Just name the numeric columns used for distributions and correlations below.

In [37]:
numeric_cols = ["alcohol", "vintage", "case_production", "retail", "rating", "bottle_size"]

df[numeric_cols].dtypes

alcohol            float64
vintage            float64
case_production    float64
retail             float64
rating               int64
bottle_size        float64
dtype: object

## 1. Missing values

Per-column null counts and percentages.

In [38]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = (
    pd.DataFrame({"nulls": missing, "pct": missing_pct})
    .query("nulls > 0")
    .sort_values("nulls", ascending=False)
)
missing_df

,nulls,pct
case_production,39158,28.96
retail,8032,5.94
vintage,4618,3.42
alcohol,1563,1.16
reviewer,244,0.18
bottle_size,29,0.02
review,17,0.01
varietal_label,12,0.01
company,2,0.00


### Missing-value heatmap

Each row = column, each column = sample of records. Red stripes show
where nulls cluster — useful for spotting structurally-missing groups
(e.g. `case_production` mostly missing for non-US wines).

In [39]:
sample = df.sample(min(5000, len(df)), random_state=42).sort_index()
fig = px.imshow(
    sample.isna().T.astype(int),
    aspect="auto",
    color_continuous_scale=["white", "tomato"],
    title="Missing-value heatmap (5k sample; red = missing)",
    labels={"x": "row sample", "y": "column", "color": "missing"},
)
fig.update_layout(height=700, coloraxis_showscale=False)
fig.show()

## 2. Retail price distribution

`retail` is the target variable. Expect strong right-skew → confirm a
`log` transform is appropriate for modelling.

In [40]:
retail = df["retail"].dropna()
print(retail.describe().round(2))
print()
print(f"Skewness (raw):    {retail.skew():.2f}")
print(f"Skewness (log1p):  {np.log1p(retail).skew():.2f}") # type: ignore
print(f"Zeros:             {(retail == 0).sum():,}")
print(f"Above $500:        {(retail > 500).sum():,}")

count    127160.00
mean         44.50
std          73.18
min           1.00
25%          20.00
50%          32.00
75%          53.00
max        9999.99
Name: retail, dtype: float64

Skewness (raw):    60.18
Skewness (log1p):  0.61
Zeros:             0
Above $500:        168


In [41]:
low, high = retail.quantile([0.01, 0.99])
fig = px.histogram(
    retail[(retail >= low) & (retail <= high)],
    nbins=100,
    title=f"Retail price — 1st-99th pct (${low:.0f}-${high:.0f})",
    labels={"value": "Retail (USD)"},
    marginal="box",
    opacity=0.85,
)
fig.update_layout(bargap=0.05, showlegend=False)
fig.show()

In [42]:
log_retail = np.log10(retail[retail > 0])
fig = px.histogram(
    log_retail,
    nbins=80,
    title="log10(retail) — much closer to normal, confirms log target",
    labels={"value": "log10(retail USD)"},
    marginal="box",
    opacity=0.85,
)
fig.update_layout(bargap=0.05, showlegend=False)
fig.show()

## 3. Rating distribution

Wine Enthusiast publishes mostly 80+ ratings — expect a narrow range
centred around ~90 with a hard floor at 80.

In [43]:
rating = df["rating"].dropna()
print(rating.describe().round(2))
print(f"Below 80: {(rating < 80).sum():,}  (likely cast errors)")

count    135192.00
mean         90.35
std           2.70
min          80.00
25%          88.00
50%          90.00
75%          92.00
max         100.00
Name: rating, dtype: float64
Below 80: 0  (likely cast errors)


In [44]:
fig = px.histogram(
    rating[rating >= 80],
    nbins=21,
    title="Rating distribution (80-100)",
    labels={"value": "Rating (pts)"},
    marginal="box",
    opacity=0.85,
)
fig.update_layout(bargap=0.05, showlegend=False)
fig.show()

## 4. Top categorical values

Cardinality and dominance check — which countries / types / varietals
dominate the dataset, and how long the long tail is.

In [45]:
cat_cols = ["country", "wine_type", "varietal_label", "state", "reviewer", "appellation"]

print(f"{'column':<18} {'unique':>8} {'top':>30} {'top_count':>10} {'top_pct':>8}")
print("-" * 80)
for col in cat_cols:
    vc = df[col].value_counts(dropna=False)
    top = vc.index[0]
    top_count = vc.iloc[0]
    top_pct = top_count / len(df) * 100
    print(f"{col:<18} {df[col].nunique():>8,} {str(top)[:30]:>30} {top_count:>10,} {top_pct:>7.1f}%")

column               unique                            top  top_count  top_pct
--------------------------------------------------------------------------------
country                  42                            USA     55,016    40.7%
wine_type                10                            Red     80,206    59.3%
varietal_label        1,047                     Pinot Noir     15,174    11.2%
state                    45                          False     79,510    58.8%
reviewer                 50                           R.V.     24,370    18.0%
appellation           1,525                     California      2,928     2.2%


In [46]:
TOP_N = 15
for col in ["country", "wine_type", "varietal_label", "reviewer"]:
    counts = df[col].value_counts().head(TOP_N)
    fig = px.bar(
        x=counts.values,
        y=counts.index,
        orientation="h",
        title=f"Top {TOP_N} {col} by review count",
        labels={"x": "count", "y": col},
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(height=420)
    fig.show()

## 5. Numeric correlations

Pearson correlations between numeric features. Look for the
`rating`-`retail` signal in particular.

In [47]:
corr = df[numeric_cols].corr(method="pearson").round(3)
fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Numeric feature correlations (Pearson)",
)
fig.update_layout(height=520, width=620)
fig.show()
corr

,alcohol,vintage,case_production,retail,rating,bottle_size
alcohol,1.000,-0.229,-0.051,0.144,0.206,0.007
vintage,-0.229,1.000,0.052,-0.150,-0.084,0.014
case_production,-0.051,0.052,1.000,-0.066,-0.109,0.090
retail,0.144,-0.150,-0.066,1.000,0.309,-0.005
rating,0.206,-0.084,-0.109,0.309,1.000,-0.043
bottle_size,0.007,0.014,0.090,-0.005,-0.043,1.000


In [48]:
# log-retail correlations — usually stronger than raw retail
df_log = df[numeric_cols].copy()
df_log["log_retail"] = np.log1p(df_log["retail"])
log_corr = df_log.corr(method="pearson")["log_retail"].drop("log_retail").round(3)
log_corr.sort_values(ascending=False)

rating             0.623
retail             0.558
alcohol            0.300
bottle_size       -0.007
case_production   -0.174
vintage           -0.253
Name: log_retail, dtype: float64

## 6. Price vs Rating

Sample-based scatter (full dataset overplots heavily). Y-axis log to
spread the long tail; colour by `wine_type`.

In [49]:
mask = df["retail"].between(0, 200) & df["rating"].notna()
sample = df[mask].sample(min(15000, mask.sum()), random_state=42)

fig = px.scatter(
    sample,
    x="rating",
    y="retail",
    color="wine_type",
    opacity=0.7,
    title="Retail vs Rating (15k sample)",
    labels={"retail": "Retail (USD)", "rating": "Rating"},
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(marker=dict(size=5, line=dict(width=0)))
fig.update_xaxes(range=[80, 100], dtick=2)
fig.update_yaxes(range=[0, 200], dtick=25)
fig.update_layout(height=560, plot_bgcolor="white")
fig.show()

In [50]:
# Mean and median retail per rating bucket
by_rating = (
    df.dropna(subset=["retail", "rating"])
    .assign(rating=lambda d: d["rating"].astype(int))
    .groupby("rating")["retail"]
    .agg(["count", "mean", "median"])
    .round(2)
)
by_rating

,count,mean,median
rating,,,
80,50,20.70,17.0
81,54,17.33,15.0
82,234,22.32,18.0
83,480,19.73,16.0
84,1192,19.44,16.0
85,2516,19.80,16.0
86,4821,21.89,17.0
87,8910,23.48,19.0
88,14743,27.69,22.0


In [51]:
fig = px.line(
    by_rating.reset_index(),
    x="rating",
    y=["mean", "median"],
    title="Retail by rating — mean vs median",
    labels={"value": "Retail (USD)", "variable": "stat"},
    markers=True,
)
fig.show()

## 7. ydata-profiling report (optional)

Runs the full profile and saves as HTML. Heavy — a couple of minutes
on 135k rows. Skip if you've already generated it.

In [ ]:
# ydata-profiling 4.18 predates pandas 3.0 and chokes on its Arrow-backed
# `str` dtype (value_counts → ArrowExtensionArray has no `.sum()`; later
# pyarrow iteration errors). Work around it without touching the rest of
# the notebook:
#   1. force python-backed string storage so ydata's internal `.astype(str)`
#      stays numpy-friendly,
#   2. cast text columns to plain `object`,
#   3. disable the chi-squared stat (the only path that calls the broken sum).
try:
    from ydata_profiling import ProfileReport

    df_prof = df.astype({c: object for c in df.columns if str(df[c].dtype) == "str"})

    with pd.option_context("mode.string_storage", "python"):
        profile = ProfileReport(
            df_prof,
            title="Wine Reviews EDA",
            explorative=True,
            vars={"num": {"chi_squared_threshold": 0.0},
                  "cat": {"chi_squared_threshold": 0.0}},
        )
        profile.to_file(r"..\..\.data\wine_reviews_silver_profile.html")
    print("Profile saved to .data/wine_reviews_silver_profile.html")
except ImportError as exc:
    print(f"ydata-profiling not available: {exc}")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 26.79it/s]

Profile saved to .data/wine_reviews_profile.html


## Key findings

- **Retail is strongly right-skewed** (skew ~10+ raw, ~0 log) — train on
  `log_retail`. A long tail of $500+ bottles will distort RMSE if kept.
- **Rating is narrow (80-100) and centred at ~90** — publication bias.
  Variance is small but its correlation with `retail` is the dominant
  numeric signal (matches the XGBoost feature-importance from Sprint 1).
- **`case_production` is 29% missing** and has an extreme max (1e8) —
  cap aggressively in Silver, or treat missing as a category.
- **`designation` is 24% missing** — high-cardinality (47k unique). Best
  handled with target encoding or "none" bucket, not one-hot.
- **USA dominates** (~37% of reviews); top 5 countries cover >80%.
- **`reviewer` has only ~50 distinct values** with heavy concentration —
  small enough for ordinal/one-hot encoding, may carry stylistic bias.
- **Numeric correlations are weak overall** except `rating`-`retail`
  (~0.4 raw, stronger on log) — most of the signal will come from
  categoricals and the review text (Sprint 5/6).